## Libraries & Function

In [ ]:
%matplotlib ipympl
import numpy as np
from scipy.linalg import expm
from qutip import *
import numba
from numba import njit, prange
import matplotlib.pyplot as plt
from IPython.display import Image, display, Math


In [ ]:

sz = np.array(([[1,0], [0,-1]]), dtype=complex); sx = np.array(([[0,1],[1,0]]), dtype=complex); sy = np.array(([[0,-1j],[1j,0]]), dtype=complex) 


In [ ]:
#funzione per plottare in LaTex delle matrici
def array_to_latex(array, real = False, array_name = None):
    array = array.real if real else array
    matrix = ''
    for row in array:
        try:
            for number in row:
                matrix += f'{number}&'
        except TypeError:
            matrix += f'{row}&'
        matrix = matrix[:-1] + r'\\'
    if array_name != None:
        display(Math(array_name+r' = \begin{bmatrix}'+matrix+r'\end{bmatrix}'))
    else:
        display(Math(r'\begin{bmatrix}'+matrix+r'\end{bmatrix}'))

### Lindblad functions

In [ ]:
def Liouvillian(H, gamma_k, L_k):
    """
    Build the Liouvillian superoperator.
    
    Parameters: - H : ndarray, Hamiltonian matrix
                - gamma_k : list, Decay rates
                - L_k : list, Jump Operators
    
    Returns: - super_L : ndarray, Liouvillian superoperator
    """    
    I = np.eye(H.shape[0])
    super_L = -1.j * (np.kron(I, H) - np.kron(H.T, I))
    
    for k in range(len(gamma_k)):
        super_L += gamma_k[k] * (np.kron(np.conj(L_k[k]), L_k[k]) - 0.5 * np.kron(I, np.conj(L_k[k]).T @ L_k[k]) - 0.5 * np.kron((np.conj(L_k[k]).T @ L_k[k]).T, I))
    
    return super_L


In [ ]:
@njit(cache=True)
def _evolve_expm_core(super_U, rho_vec_initial, n_times):
    """
    Core evolution loop with expm method (Numba JIT)
    """
    rho_size = rho_vec_initial.shape[0]
    rho_vec_list = np.zeros((rho_size, n_times), dtype=np.complex128)
    rho_vec_list[:, 0] = rho_vec_initial
    
    for i in range(1, n_times):
        rho_vec_list[:, i] = super_U @ rho_vec_list[:, i - 1]
    
    return rho_vec_list


@njit(cache=True)
def _evolve_diagonal_core(V, V_inv, U_diag, rho_vec_initial, n_times):
    """
    Core evolution loop with diagonal method (Numba JIT)
    """
    n_states = len(U_diag)
    
    # Initial coefficients in eigenbasis
    coeff = V_inv @ rho_vec_initial
    coeff_list = np.zeros((n_states, n_times), dtype=np.complex128)
    coeff_list[:, 0] = coeff
    
    # Evolution of coefficients
    for i in range(1, n_times):
        coeff_list[:, i] = U_diag * coeff_list[:, i - 1]
    
    # Transform back to original basis
    rho_vec_list = V @ coeff_list
    
    return rho_vec_list


def Lindblad_evo(rho, H, gamma_k, L_k, times, method="expm", vectorized=True):
    """
    Evolution of the density matrix with the Lindblad Eq. (Optimized with Numba)
    
    Method: - "expm" -> propagator = expm(super_L * dt)
            - "diagonal" -> diagonalization of the super-operator
        
    Vectorized: True/False to choose the output format
    
    Parameters: - H : Qobj or ndarray, System Hamiltonian
                - rho : Qobj or ndarray, Initial Density Matrix
                - gamma_k : list, List of Decay Rates
                - L_k : list, List of Jump Operators
                - times : array, Time array
        
    Returns : - if vectorized=True → array (N^2, Nt)
              - if vectorized=False → array (Nt, N_site, N_site)
              - if method="diagonal" also returns V, W
    """
    # Convert to NumPy
    L_k = [L.full() if hasattr(L, "full") else np.array(L, dtype=complex) for L in L_k]
    H = H.full() if hasattr(H, "full") else np.array(H, dtype=complex)
    rho = rho.full() if hasattr(rho, "full") else np.array(rho, dtype=complex)
    
    rho_shape = H.shape[0]
    dt = times[1] - times[0]
    
    # Build Liouvillian
    super_L = Liouvillian(H, gamma_k, L_k)
    
    # Vectorize initial state
    rho_vec = rho.reshape(rho_shape * rho_shape)
    
    # -------------
    # Expm method
    # -------------
    if method == "expm":
        # Compute propagator 
        super_U = expm(super_L * dt)
        
        # evolution loop
        rho_vec_list = _evolve_expm_core(super_U, rho_vec, len(times))
        
        # Output
        if vectorized:
            return rho_vec_list
        else:
            return rho_vec_list.T.reshape(len(times), rho_shape, rho_shape)
    
    # ------------------
    # Diagonal method
    # ------------------
    elif method == "diagonal":
        # Diagonalize Liouvillian 
        W, V = np.linalg.eig(super_L)
        V_inv = np.linalg.inv(V)
        
        # Diagonal evolution operator
        U_diag = np.exp(W * dt)
        
        # evolution loop
        rho_vec_list = _evolve_diagonal_core(V, V_inv, U_diag, rho_vec, len(times))
        
        # Output
        if vectorized:
            return rho_vec_list, V, W
        else:
            return rho_vec_list.T.reshape(len(times), rho_shape, rho_shape), V, W
    
    else:
        raise ValueError("method must be 'expm' or 'diagonal'")

### Collisional Method functions

In [ ]:
def system_Hamiltonian(N_site, E, V_pot, mode="complete"):
    """
    Build up of the System's Hamiltonian for the complete basis (ground & excited states) or only excited states.
    
    Method: - "complete"-> complete basis (ground & excited states)
            - "exc"-> excited basis (only excited states)
    
    Parameters: - E: Float, System's Site Energies (randomly generated)
                - V_pot: Float, Hopping Potential
                - N_site : Int, Number of Sites
        
    Returns : System's Hamiltonian as Numpy array
    """
    # -------------------------
    # Only Excited States Basis
    # -------------------------
    if mode == "exc":   
        H_sys = np.zeros((N_site, N_site), dtype=complex)
        for i in range(N_site):
            H_sys[i, i] = E[i]
        for i in range(N_site):
            for j in range(N_site):
                if i != j:
                    H_sys[i, j] = V_pot
        return H_sys
        
    # --------------
    # Complete Basis 
    # --------------    
    elif mode == "complete":   
        H_sys = np.zeros((2**N_site, 2**N_site), dtype='complex')
        
        for i in range(N_site):
            H_i = (E[i]/2) * (tensor(identity(2**i), identity(2)-sigmaz(), identity(2**(N_site-i-1)))) # g states have 0 energy and the e sites have all the energy
            H_sys += H_i.full()

            for j in range(i+1, N_site):
               H_ij = V_pot/2 * (tensor(identity(2**i), sigmax(), identity(2**(j-i-1)), sigmax(), identity(2**(N_site-j-1))) + tensor(identity(2**i), sigmay(), identity(2**(j-i-1)), sigmay(), identity(2**(N_site-j-1))))
               H_sys += H_ij.full()
        
        return H_sys

    else:
        raise ValueError("mode has to be 'complete' or 'exc'")


In [ ]:
def interaction_Hamiltonian(N_site, c_CM, method='QJ'):   #Hamiltonian of interaction with N different ancillas, one for every site of the system, in the Quantum Jump Limit
    """
    Build up of the Hamiltonian of Interaction for the Collision System - Ancilla in both Quantum Jump and Diffusive Limit
       
    Parameters: - N_site : int, Number of Sites
                - c_CM : list, Interaction Forces for the System - Ancilla intercation/collsion

    Method: - QJ : Quantum Jump Limit
            - Diff : Diffusive Limit
        
    Returns : Hamiltonian of Interaction as Qutip object
    """
    dim_tot = 2**(2 * N_site)
    H_int = np.zeros((dim_tot, dim_tot), dtype=complex)   #inizialization

    # Selection of the Ancilla's operator
    if method == 'QJ':
        anc_op = sigmax() # Interazione Z (sys) - X (anc) -> induce salti/flip
    elif method == 'Diff':
        anc_op = sigmaz() # Interazione Z (sys) - Z (anc) -> induce dephasing
    else:
        raise ValueError("Method has to be 'QJ' or 'Diff'")
        
    for j in range(N_site):
  
        op_list = [identity(2) for _ in range(2 * N_site)]  #list of identity to be fill with the operator sigmaz & sigmax; 2N identity, N for the system and N fo the ancillas
            
        op_list[j] = sigmaz()      # sigmaz acts on the j site
        op_list[N_site + j] = anc_op  # sigmax acts on the j ancilla, with index N + j
            
        H_term = c_CM[j] * tensor(op_list)  # tensor product between the element of the list
            
        H_int += H_term.full()
    
    return H_int


In [ ]:
def hamiltonian_N_ancillas(N_site, E, V_pot, c_CM, int_type="QJ"):
    """
    Generation of 3 Hamiltonians for the collision model with N ancillas:
                - H_system : system Hamiltonian
                - H_collision : interaction Hamiltonian with N ancillas
                - H_tot : complete Hamiltonian (system + collision)

    Parameters: - E: Float, System's Site Energies (randomly generated)
                - V_pot: Float, Hopping Potential
                - N_site : int, Number of Sites
                - c_CM : list, Interaction Forces for the System - Ancilla intercation/collsion
    
    Interaction_type : - QJ : Quantum Jump Limit
                       - Diff : Diffusive Limit

    Returns : H_system, H_collision, H_tot
    """
    
    H_collision = interaction_Hamiltonian(N_site, c_CM, method = int_type) 
    
    H_system = system_Hamiltonian(N_site, E, V_pot, mode="complete")
        
    dim_anc = 2**N_site
    Id_ancillas = np.eye(dim_anc, dtype=complex)
    H_system_expanded = np.kron(H_system, Id_ancillas)  #expand H_sys in the total space
        
    H_tot = H_system_expanded + H_collision
        
    return H_system, H_collision, H_tot


In [ ]:
def evolution_operator(H, dt, method='expm', hermitian=True):
    """
    Build up of the evolution operator U = exp(-i H dt) using Expm or analytic diagonalization.
   
    Parameters: - H : Qobj or ndarray, System Hamiltonian
                - dt : float, Timestep
    
    Method : - "expm"-> build up of the Matrix Exponential with expm
             - "diagonalization"->  build up of the propagater U as V @(exp(-i W dt))@ V_dag with W eigenvalues and V eigenvector of the Hamiltonian 

    Returns : Evolution Operator U, 
    """
    H = H.full() if hasattr(H, "full") else np.array(H)
    
    # -----------
    # Expm method
    # -----------
    
    if method == 'expm':
        U = expm(-1j * H * dt)
        return U
        
    # ---------------
    # Diagonalization
    # ---------------
    
    elif method == 'diagonalization':
        if hermitian:
            w, V = np.linalg.eigh(H)
            V_inv = V.conj().T
        else:
            w, V = np.linalg.eig(H) 
            V_inv = np.linalg.inv(V)
                
        U_diag = np.diag(np.exp(-1j * w * dt))
        U = V @ U_diag @ V_inv
        return U, U_diag, w, V

    else:
        raise ValueError("method must be 'expm' or 'diagonalization'")


<h1 style="color:red;">Case of Study : Exciton Dimer</h1> 

#### Evolution with $ U_{complete} $ and then trace on the ancilla

In [ ]:
@njit(cache=True)
def _compute_trace_ancilla_core_general(rho_sys, rho_anc, U_step, U_step_dag, projectors, times, dim_sys, dim_anc, N_site):
    """
    Core computation optimized with Numba - generalized for N sites
    """
    pops_complete = np.zeros((N_site, len(times)), dtype=np.float64)
    
    # Initial state - all sites
    for site in range(N_site):
        pops_complete[site, 0] = np.real(np.trace(projectors[site] @ rho_sys))
    
    # Time Evolution
    for t in range(1, len(times)):
        # 1: Expansion
        rho_tot = np.kron(rho_sys, rho_anc)
        
        # 2: Evolution
        rho_tot = U_step @ rho_tot @ U_step_dag
        
        # 3: Partial Trace
        rho_tot_reshaped = rho_tot.reshape(dim_sys, dim_anc, dim_sys, dim_anc)
        
        # Manual trace
        rho_sys = np.zeros((dim_sys, dim_sys), dtype=np.complex128)
        for i in range(dim_sys):
            for j in range(dim_sys):
                for k in range(dim_anc):
                    rho_sys[i, j] += rho_tot_reshaped[i, k, j, k]
        
        # 4: Store populations - all sites
        for site in range(N_site):
            pops_complete[site, t] = np.real(np.trace(projectors[site] @ rho_sys))
    
    return pops_complete


def compute_trace_ancilla(rho_sys_initial, U_diag, V, times, projectors, N_site, method='QJ'):
    """
    Generalized version with Numba optimization for arbitrary number of sites.
    Automatically builds the correct ancilla density matrix based on 'method'.

    Parameters: - method: - QJ : Quantum Jump Limit, uses |0><0|
                          - Diff : Diffusive Limit, uses I/2
    """
    
    if method == 'QJ':
        
        rho_anc_single = basis(2,0) # Pure state |0><0|: [[1, 0], [0, 0]]
    
    elif method == 'Diff':
        
        rho_anc_single = (qeye(2) / 2) # Completely Mixed state I/2: [[0.5, 0], [0, 0.5]]
    else:
        raise ValueError("Method must be 'QJ' or 'Diff'")

    rho_anc = (tensor([rho_anc_single for _ in range(N_site)])).full() #for N ancilla
    
    # Convert to numpy
    rho_sys = rho_sys_initial.full() if hasattr(rho_sys_initial, 'full') else rho_sys_initial.copy()

    # Dimensions
    dim_sys = rho_sys.shape[0]
    dim_anc = rho_anc.shape[0]
    
    # Evolution operator
    V_np = V.full() if hasattr(V, 'full') else V
    U_diag_np = U_diag.full() if hasattr(U_diag, 'full') else U_diag
    U_step = V_np @ U_diag_np @ V_np.conj().T
    U_step_dag = U_step.conj().T
    
    # Call JIT-compiled function
    pops_complete = _compute_trace_ancilla_core_general(
        rho_sys, rho_anc, U_step, U_step_dag, 
        projectors, times, dim_sys, dim_anc, N_site )
    
    return pops_complete

#### Single trajectory Quantum Jump

In [ ]:
@njit(parallel=True, cache=True, fastmath=True)
def _compute_trajectory_wf_core_QJ(psi_sys_initial, U_site, Sz_ops, projectors,
                                      jump_probabilities, N_traj, N_site, n_times, seed=None):
    """
    Core computation optimized with Numba (JIT compiled with parallelization)
    """
    if seed is not None:
        np.random.seed(seed)
    
    pop_traj = np.zeros((N_site, n_times, N_traj), dtype=np.float64)
    count = np.zeros(N_traj, dtype=np.float64)
    
    # Precompute initial populations for all sites
    for site in range(N_site):
        P_psi = projectors[site] @ psi_sys_initial
        pop_initial = np.real(np.vdot(psi_sys_initial, P_psi))
        for traj in range(N_traj):
            pop_traj[site, 0, traj] = pop_initial
    
    # Parallel loop over trajectories
    for traj in prange(N_traj):
        psi = psi_sys_initial.copy()
        
        for step in range(1, n_times):
            psi = U_site @ psi
            
            # Monte Carlo jumps
            for site_index in range(N_site):
                if np.random.rand() < jump_probabilities[site_index]:
                    psi = Sz_ops[site_index] @ psi
                    count[traj] += 1
            
            # Compute populations for all sites
            for site in range(N_site):
                P_psi = projectors[site] @ psi
                pop_traj[site, step, traj] = np.real(np.vdot(psi, P_psi))
    
    return pop_traj, count


def compute_trajectory_wf_QJ(dt, c_CM, N_traj, N_site, times, projectors, psi_sys_initial, U_site):
    """
    Compute quantum trajectory evolution with collisional algorithm, in the Quantum Jump limit (sz x sx)
    
    Parameters: - dt : float, Time Step
                - c_CM : array, Collisional model Coefficients
                - N_traj : int, Number of Trajectories
                - N_site : int, Number of Sites
                - times : array, Time array
                - projectors : list/array, Projection Operators [P_10, P_01, ...]
                - psi_sys_initial : Qobj or ndarray, Initial Wave Function
                - U_site : Qobj or ndarray, Time Evolution Operator
                       
    Returns: - pop_traj : ndarray, Population for each trajectory (N_site x steps x N_traj)
             - average_pop_traj : ndarray, Average population over trajectories (N_site x steps)
             - count : ndarray, Number of collisions per trajectory
    """   
    # Check and conversion to Numpy
    U_site = U_site.full() if hasattr(U_site, 'full') else np.array(U_site, dtype=complex)
    psi_sys_initial = psi_sys_initial.full() if hasattr(psi_sys_initial, 'full') else np.array(psi_sys_initial, dtype=complex)
    
    # Flatten wave function if needed
    if psi_sys_initial.ndim > 1:
        psi_sys_initial = psi_sys_initial.flatten()
    
    # Convert projectors (already numpy if defined with np.kron)
    projectors = np.array([P.full() if hasattr(P, 'full') else np.array(P, dtype=complex) for P in projectors])
    
    # Evolution probabilities
    jump_probabilities = np.array([np.sin(c * dt)**2 for c in c_CM])
    
    # Construction of sigmaz operators for every system's site
    Sz_ops = []
    for idx in range(N_site):
        ops = [qeye(2) for _ in range(N_site)]
        ops[idx] = sigmaz()
        Sz_op = tensor(ops).full()
        Sz_ops.append(Sz_op)
    Sz_ops = np.array(Sz_ops)
    
    # Call JIT-compiled core function
    pop_traj, count = _compute_trajectory_wf_core_QJ(
        psi_sys_initial_np, U_site_np, Sz_ops, projectors_np,
        jump_probabilities, N_traj, N_site, len(times), seed=None )
    
    # Average over trajectories
    average_pop_traj = np.mean(pop_traj, axis=2)
    
    return pop_traj, average_pop_traj, count

#### Count number of effective collision

In [ ]:
@njit(cache=True)
def compute_count(count, steps):
    """
    Normalization of the count of the Effective collision over the number of time steps and calculation of the average probability of collsion

    Parameters: - count : array, Number of Effective Collision (that applies identity to the system wave function)
                - steps : int, number of Time Steps
                
    Returns : - count : same matrix Normalized and with number of Effective Collision (that applies sigmaz to the system wave function)
              - media : average probabilty to have an Effective Collision calculated over N trajectories
    """
    count_normalized = count / steps
    media = np.mean(count_normalized)
    
    return count_normalized, media

#### Single trajectory Diffusional

In [ ]:
@njit(parallel=True, cache=True, fastmath=True)
def _compute_trajectory_wf_diff_core(psi_sys_initial, U_site, Sz_ops, projectors,
                                      c_CM, dt, N_traj, N_site, n_times, seed=None):
    """
    Core computation for diffusive limit (Numba JIT with parallelization)
    """
    if seed is not None:
        np.random.seed(seed)
    
    pop_traj = np.zeros((N_site, n_times, N_traj), dtype=np.float64)
    
    # Precompute trigonometric values
    cos_vals = np.cos(c_CM * dt)
    sin_vals = np.sin(c_CM * dt)
    
    # Precompute initial populations for all sites
    for site in range(N_site):
        P_psi = projectors[site] @ psi_sys_initial
        pop_initial = np.real(np.vdot(psi_sys_initial, P_psi))
        for traj in range(N_traj):
            pop_traj[site, 0, traj] = pop_initial
    
    # Parallel loop over trajectories
    for traj in prange(N_traj):
        psi = psi_sys_initial.copy()
        
        for step in range(1, n_times):
            psi = U_site @ psi
            
            for site_index in range(N_site):
                Sz_psi = Sz_ops[site_index] @ psi
                
                if np.random.rand() < 0.5:
                    psi = cos_vals[site_index] * psi - 1j * sin_vals[site_index] * Sz_psi
                else:
                    psi = cos_vals[site_index] * psi + 1j * sin_vals[site_index] * Sz_psi

            # Normalization
            psi = psi / np.linalg.norm(psi)

            # Measurament
            for site in range(N_site):
                P_psi = projectors[site] @ psi
                pop_traj[site, step, traj] = np.real(np.vdot(psi, P_psi))
    
    return pop_traj

def compute_trajectory_wf_Diff(dt, c_CM, N_traj, N_site, times, projectors, psi_sys_initial, U_site):
    """
    Compute quantum trajectory evolution with collisional algorithm in the Diffusive limit (sz x sz).
    
    Parameters: - dt : float, Time Step
                - c_CM : array, Collisional model Coefficients
                - N_traj : int, Number of Trajectories
                - N_site : int, Number of Sites
                - times : array, Time array
                - projectors : list/array, Projection Operators [P_10, P_01, ...]
                - psi_sys_initial : Qobj or ndarray, Initial Wave Function
                - U_site : Qobj or ndarray, Time Evolution Operator
                       
    Returns: - pop_traj : ndarray, Population for each trajectory (N_site x steps x N_traj)
             - average_pop_traj : ndarray, Average population over trajectories (N_site x steps)
    """   
    # Convert to Numpy
    U_site = U_site.full() if hasattr(U_site, 'full') else np.array(U_site, dtype=complex)
    psi_sys_initial = psi_sys_initial.full() if hasattr(psi_sys_initial, 'full') else np.array(psi_sys_initial, dtype=complex)
    
    if psi_sys_initial.ndim > 1:
        psi_sys_initial = psi_sys_initial.flatten()
    
    projectors = np.array([
        P.full() if hasattr(P, 'full') else np.array(P, dtype=complex)
        for P in projectors])
    
    c_CM = np.array(c_CM, dtype=np.float64)  # ← Cambiato
    
    # Sigmaz operators
    Sz_ops = []
    for idx in range(N_site):
        ops = [qeye(2) for _ in range(N_site)]
        ops[idx] = sigmaz()
        Sz_ops.append(tensor(ops).full())
    Sz_ops = np.array(Sz_ops)
    
    # Call JIT function
    pop_traj = _compute_trajectory_wf_diff_core(
        psi_sys_initial, U_site, Sz_ops, projectors,
        c_CM, dt, N_traj, N_site, len(times), seed=None)
    
    average_pop_traj = np.mean(pop_traj, axis=2)
    
    return pop_traj, average_pop_traj
    

#### Isolated system

In [ ]:
@njit(cache=True)
def _compute_trajectory_isolated_core_general(psi_initial, U_site, projectors, n_times):
    """
    Core evolution loop - general for any number of sites
    """
    N_site = projectors.shape[0]
    pop_traj = np.zeros((N_site, n_times), dtype=np.float64)
    
    # Initial populations
    for site in range(N_site):
        P_psi = projectors[site] @ psi_initial
        pop_traj[site, 0] = np.real(np.vdot(psi_initial, P_psi))
    
    # Evolution
    psi = psi_initial.copy()
    for step in range(1, n_times):
        psi = U_site @ psi
        
        for site in range(N_site):
            P_psi = projectors[site] @ psi
            pop_traj[site, step] = np.real(np.vdot(psi, P_psi))
    
    return pop_traj

def compute_trajectory_wf_isolated(N_site, times, projectors, psi_sys_initial, U_site):
    """
    Optimized isolated system evolution with Numba (works for any N_site).
    """
    # Convert to NumPy
    U_site_np = U_site.full() if hasattr(U_site, 'full') else np.array(U_site, dtype=complex)
    psi_initial_np = psi_sys_initial.full() if hasattr(psi_sys_initial, 'full') else np.array(psi_sys_initial, dtype=complex)
    
    # Flatten if needed
    if psi_initial_np.ndim > 1:
        psi_initial_np = psi_initial_np.flatten()
        
    # JIT-compiled evolution
    pop_traj_isolated = _compute_trajectory_isolated_core_general(
        psi_initial_np, U_site_np, projectors, len(times))
    
    return pop_traj_isolated


#### Lindblad

In [ ]:
def compute_Lindblad(H_system, rho_sys_initial, gamma_k, L_k, times):
    """
    Compute Lindblad evolution using diagonal method.
    
    Parameters: - H_system : Qobj or ndarray, System Hamiltonian
                - rho_sys_initial : Qobj or ndarray, Initial Density Matrix
                - gamma_k : list, List of Decay Rates
                - L_k : list, List of Jump Operators
                - times : array, Time array
    
    Returns: - rho_list_lindblad : ndarray, Density matrix evolution (len(times) x N_site x N_site)
             - V : ndarray, Eigenvectors of the Liouvillian
             - W : ndarray, Eigenvalues of the Liouvillian
    """
    
    rho_list_lindblad, V, W = Lindblad_evo(rho_sys_initial, H_system, gamma_k, L_k, times, method="diagonal", vectorized=False)

    return rho_list_lindblad, V, W


## Main Loop for varying $ dt $ and $ N_{traj} $

In [ ]:
# System's Parameters
N_site = 2            # Number of sites
V_pot = 1.0           # Hopping Potential
E = 1.5 + np.random.randn(N_site)*0.1     #random inizialization of the system energies

# Time Evolution Parameters
dt_list = [0.01, 0.02, 0.05, 0.1 ]     # Time step
tf = 50.0    # Final Time
steps_list = [ int(tf / dt_list[i]) for i in range (len(dt_list)) ]
times_list = [ np.linspace(0, tf, int(steps_list[i])) for i in range(len(dt_list))]

N_traj_list = [100, 1000, 10000]

# Dephasing Parameter (come in MATLAB)
g_deph = 0.1       # Gamma rate

# Scaling for the collsional algorithm c = sqrt(gamma / 4dt)
c_CM_list = np.array([[np.sqrt(g_deph / (4 * dt_list[j])) for j in range(len(dt_list))] for _ in range(N_site)])  # same Coupling for the 2 sites

#Lindblad Rates
gamma_k = [g_deph, g_deph]

#array_to_latex(c_CM)
#array_to_latex(E)

In [ ]:
# Initial wave function and density matrix
# System
psi_sys_initial = tensor(basis(2, 0), basis(2, 1)) # I set the population only in site 2
rho_sys_initial = (ket2dm(psi_sys_initial)).full()

# Ancilla QJ
psi_anc_single_QJ = basis(2, 0)
rho_anc_single_QJ = ket2dm(psi_anc_single_QJ ) # Pure state |0><0| for a singe ancilla 
rho_anc_all_QJ = (tensor([rho_anc_single_QJ for _ in range(N_site)])).full() #for N ancilla

# Ancilla Diff
psi_anc_single_Diff_0 = (basis(2, 0)).full() # Pure state |0><0| for a singe ancilla
psi_anc_single_Diff_1 = (basis(2, 1)).full() # Pure state |1><1| for a singe ancilla
rho_anc_single_Diff = qeye(2) / 2 # Completely Mixed  1/2 (Identity) 
rho_anc_all_Diff = (tensor([rho_anc_single_Diff for _ in range(N_site)])).full() #for N ancilla

P0 = (np.eye(2, dtype=complex) + sz) / 2 # projector on |0>
P1 = (np.eye(2, dtype=complex) - sz) / 2 # projector su |1>

P_00 = np.kron(P0, P0) # |00><00|
P_01 = np.kron(P0, P1) # |01><01|
P_10 = np.kron(P1, P0) # |10><10|
P_11 = np.kron(P1, P1) # |11><11|

projectors = np.array([P_10, P_01], dtype=complex) # for only excited states

#Lindblad Jump Operator
L_1 = P_10  # projector on |10><10|
L_2 = P_01  # projector on |01><01|
L_k = [L_1, L_2]


### Calculation

In [ ]:
import time   # to evaluate the duration of every calculation

# Dictionary for results
results = {}

print("Starting computation for different dt and N_traj")

# Loop over dt
for dt_idx, dt in enumerate(dt_list):

    print("=" * 40)
    print(f"Processing dt = {dt:.4f} ({dt_idx+1}/{len(dt_list)})")
    

    # Initialize dictionary for this dt
    results[dt] = {}
    
    # Extract parameters for this dt
    times = times_list[dt_idx]
    steps = steps_list[dt_idx]
    c_CM = c_CM_list[:, dt_idx]  # Column corresponding to this dt
    
    # =====================================
    # Recalculate Hamiltonian and Operators
    # =====================================
    print(f"Recalculating Hamiltonians for dt = {dt:.4f}")       #because c_CM depends on dt
    
    # Hamiltonian & U for Density Matrix (with ancillas)
    H_site, H_coll, H_tot = hamiltonian_N_ancillas(N_site, E, V_pot, c_CM, int_type="Diff")  #change : int_type
    
    U_tot, U_diag, w, V = evolution_operator(H_tot, dt, method='diagonalization', hermitian=True)
    U_diag_dag = U_diag.conj().T; V_dag = V.conj().T
    
    # Hamiltonian & U for Wave Function (system only)
    H_system = system_Hamiltonian(N_site, E, V_pot, mode="complete")
    
    U_site, U_diag_site, w_site, V_site = evolution_operator(H_system, dt, method='diagonalization', hermitian=True)
        
    # =========
    # Lindblad
    # =========
    print("Computing Lindblad")
    start_time = time.time()
    
    rho_list_lindblad, V_lindblad, W_lindblad = compute_Lindblad(H_system, rho_sys_initial, gamma_k, L_k, times)
    
    lindblad_time = time.time() - start_time
    
    print(f"Completed in {lindblad_time:.2f}s")
    
    # ==============
    # Trace Ancilla
    # ==============
    print(f"Computing Trace Ancilla")
    start_time = time.time()
    
    pops_trace = compute_trace_ancilla(rho_sys_initial, U_diag, V, times, projectors, N_site, method='Diff') # change : method
    
    trace_time = time.time() - start_time
    
    print(f"Completed in {trace_time:.2f}s")

    # ========================================
    # Trajectory Isolated (without collisions)
    # ========================================
    print(f"Computing Trajectory Isolated")
    start_time = time.time()
        
    pop_traj_isolated = compute_trajectory_wf_isolated(N_site, times, projectors, psi_sys_initial, U_site)
        
    isolated_time = time.time() - start_time
    
    print(f"Completed in {isolated_time:.2f}s")
    
    # ================
    # Loop over N_traj
    # ================
    for N_traj in N_traj_list:

        print("-" * 40)
        print(f"N_traj = {N_traj}")
        
        # Initialize dictionary for this N_traj
        results[dt][N_traj] = {}
        
        # ===================================
        # Trajectory for WF (with collisions)
        # ===================================
        print(f"Computing Trajectory WF")
        start_time = time.time()
                                                 # change : add count to store result and Diff with QJ
        pop_traj, average_pop_traj = compute_trajectory_wf_Diff(dt, c_CM, N_traj, N_site,        
                                                     times, projectors, psi_sys_initial, U_site)
        traj_time = time.time() - start_time
        
        print(f"Completed in {traj_time:.2f}s")
        
        # Calculate collision statistics
        #count_normalized, collision_prob = compute_count(count.copy(), steps) # change : remove '#' for QJ
        
        # ==================================
        # Save the Results in the Dictionary
        # ==================================

        n_samples_to_save = 3 # number of single exemplary trajectories to store
        indices = np.random.choice(pop_traj.shape[2], n_samples_to_save, replace=False) # random extraction of indices
        traj_samples = pop_traj[:, :, indices]
        
        results[dt][N_traj] = {
            'parameters': {
                'dt': dt,
                'N_traj': N_traj,
                'times': times,
                'steps': steps,
                'c_CM': c_CM.copy()  
            },
            
            # Trace Ancilla 
            'pops_trace': pops_trace,

             # Trajectory Isolated
            'trajectory_isolated': pop_traj_isolated,
            
            # Lindblad 
            'lindblad': {
                'rho_list': rho_list_lindblad,
                'V': V_lindblad,
                'W': W_lindblad
            },

            # Trajectory WF
            'trajectory_wf': {
                'pop_traj_samples': traj_samples,  # (N_site x len(times) x n_samples_to_save)
                'average_pop': average_pop_traj,   # (N_site x len(times))
                #'count': count,                   # change : remove '#' fo QJ
                #'count_normalized': count_normalized,
                #'collision_prob': collision_prob
            }        
        }
        
        print(f"Results saved in results[{dt}][{N_traj}]")

print("\n" + "=" * 40)
print("COMPUTATION COMPLETED!")
print(f" Results saved for:")
print(f"   - {len(dt_list)} dt values: {dt_list}")
print(f"   - {len(N_traj_list)} N_traj values: {N_traj_list}")
print("=" * 40)

In [ ]:
import pickle
import os

# Crea una cartella per i risultati (opzionale)
results_dir = "results"
#os.makedirs(results_dir, exist_ok=True)

# Salva il dizionario
filename = os.path.join(results_dir, "results_Diffusive_dt_Ntraj.pkl")  # change : filename for results

with open(filename, 'wb') as f:
    pickle.dump(results, f)

print(f"Results saved to: {filename}")
print(f"File size: {os.path.getsize(filename) / (1024**2):.2f} MB")

## Plot

In [ ]:
# Scegli quali dt e N_traj vuoi plottare
dt_plot = 0.01  # oppure 0.02, 0.05, 0.1
N_traj_plot_10000 = 10000  # oppure 1000
N_traj_plot_1000 = 1000 
N_traj_plot_100 = 100 

# Estrai i dati dal dizionario
times_plot_10000 = results[dt_plot][N_traj_plot_10000]['parameters']['times']
trace_pop_10000 = results[dt_plot][N_traj_plot_10000]['pops_trace']
average_pop_10000 = results[dt_plot][N_traj_plot_10000]['trajectory_wf']['average_pop']

times_plot_1000 = results[dt_plot][N_traj_plot_1000]['parameters']['times']
trace_pop_1000 = results[dt_plot][N_traj_plot_1000]['pops_trace']
average_pop_1000 = results[dt_plot][N_traj_plot_1000]['trajectory_wf']['average_pop']

times_plot_100 = results[dt_plot][N_traj_plot_100]['parameters']['times']
trace_pop_100 = results[dt_plot][N_traj_plot_100]['pops_trace']
average_pop_100 = results[dt_plot][N_traj_plot_100]['trajectory_wf']['average_pop']

rho_lindblad = results[dt_plot][N_traj_plot_10000]['lindblad']['rho_list']

# Plot per il sito 1 (|10>)
fig01, ax = plt.subplots(figsize=(10, 5))

ax.plot(times_plot_1000, trace_pop_1000[0, :], label=r'Trace Ancilla Site 1', linewidth=2, linestyle='--')
ax.plot(times_plot_10000, average_pop_10000[0, :], label=r'Average Trajectory Site 1 N = 10000', linewidth=2)
ax.plot(times_plot_1000, average_pop_1000[0, :], label=r'Average Trajectory Site 1 N = 1000', linewidth=2)
ax.plot(times_plot_100, average_pop_100[0, :], label=r'Average Trajectory Site 1 N = 100', linewidth=2)
ax.plot(times_plot_10000, np.real(rho_lindblad[:, 2, 2]), label=r'Lindblad Site 1', linewidth=2, linestyle=':')

ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Population', fontsize=12)
ax.set_title(f'Comparison: dt = {dt_plot}, comparison N_traj ', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
#ax.set_xlim(35, 45)
#ax.set_ylim(0.3, 0.7)
plt.show()

In [ ]:
# Plot per il sito 1 (|10>)
fig02, ax = plt.subplots(figsize=(10, 5))

# Trace Ancilla (uguale per tutti i dt, uso 0.01)
ax.plot(results[0.01][100]['parameters']['times'], results[0.01][100]['pops_trace'][0, :], 
        label=r'Trace Ancilla Site 1', linewidth=2, linestyle='--')

# Average trajectories per diversi dt
ax.plot(results[0.01][100]['parameters']['times'], results[0.01][100]['trajectory_wf']['average_pop'][0, :], 
        label=r'Average Trajectory Site 1 dt = 0.01', linewidth=2)
ax.plot(results[0.02][100]['parameters']['times'], results[0.02][100]['trajectory_wf']['average_pop'][0, :], 
        label=r'Average Trajectory Site 1 dt = 0.02', linewidth=2)
ax.plot(results[0.05][100]['parameters']['times'], results[0.05][100]['trajectory_wf']['average_pop'][0, :], 
        label=r'Average Trajectory Site 1 dt = 0.05', linewidth=2)
ax.plot(results[0.10][100]['parameters']['times'], results[0.10][100]['trajectory_wf']['average_pop'][0, :], 
        label=r'Average Trajectory Site 1 dt = 0.10', linewidth=2)

# Lindblad (uso dt=0.01, uguale per tutti)
ax.plot(results[0.01][100]['parameters']['times'], np.real(results[0.01][100]['lindblad']['rho_list'][:, 2, 2]), 
        label=r'Lindblad Site 1', linewidth=2, linestyle=':')

ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Population', fontsize=12)
ax.set_title(f'Comparison different dt (N_traj = 100)', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
#ax.set_xlim(25, 45)
#ax.set_ylim(0.3, 0.7)
plt.show()

In [ ]:
times_1000 = results[0.01][1000]['parameters']['times']
pop_trace_1000 = results[0.01][1000]['pops_trace']
pop_avg_traj_1000 = results[0.01][1000]['trajectory_wf']['average_pop']
rho_lindblad_1000 = results[0.01][1000]['lindblad']['rho_list']

times_100 = results[0.01][100]['parameters']['times']
pop_trace_100 = results[0.01][100]['pops_trace']
pop_avg_traj_100 = results[0.01][100]['trajectory_wf']['average_pop']
rho_lindblad_100 = results[0.01][100]['lindblad']['rho_list']


fig03, ax = plt.subplots(figsize=(10, 5))

ax.plot(times_100, pop_trace_100[0,:], label=r'Trace Ancilla Site 1', linewidth=2, linestyle='--')
ax.plot(times_100, np.real(rho_lindblad_100[:, 2, 2]), label=r'Linblad Site 1', linewidth=2, linestyle=':')
ax.plot(times_100, pop_avg_traj_100[0, :])
ax.legend(fontsize=10)
ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Population', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig04, ax = plt.subplots(figsize=(10,5))
ax.plot(results[0.01][1000]['parameters']['times'], results[0.01][1000]['trajectory_wf']['pop_traj_samples'][1, :, 2], label=r'TrajDiff', linewidth=2, linestyle='-')
ax.plot(results[0.01][1000]['parameters']['times'], results[0.01][1000]['trajectory_isolated'][1], label=r'Traj Isolated', linewidth=2, linestyle='--')
ax.legend(fontsize=10)
ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Population', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()